In [1]:
%pip install -q ollama==0.6.2 pypdf==6.14.2 numpy==1.26.4

from pathlib import Path
import os
import sys

# VS Code may start in the workspace root or beside this notebook.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "demos" / "3.1-model-adaptation").is_dir())
DEMO = ROOT / "demos" / "3.1-model-adaptation"
os.chdir(DEMO)
sys.path.insert(0, str(ROOT / "demos"))
sys.path.insert(0, str(DEMO))


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from notebook_setup import local_client, course_pdf
client = local_client("gemma3:12b", "nomic-embed-text")
PDF = course_pdf(ROOT)

Local model: gemma3:12b | digest: f4031aab637d1ffa37b42570452ae0e4fad0314754d17ded67322e4b95836f8a
Local model: nomic-embed-text:latest | digest: 0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f


In [3]:
import numpy as np
import ollama
from pypdf import PdfReader

CHAT_MODEL = "gemma3:12b"
EMBED_MODEL = "nomic-embed-text"

QUESTION = "How is UNSW COMP3110 assessed?"

def ask(question, context=None):
    if context is None:
        prompt = question
    else:
        prompt = (
            "Answer the question using only the context below. "
            "Cite supporting source/page labels in square brackets. "
            "If the context does not contain the answer, say so. Absence is not proof that a rule does not exist.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}"
        )
    stream = client.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 400},
        stream=True,
    )
    for part in stream:
        print(part["message"]["content"], end="", flush=True)

## No retrieval

Predict the answer. Without supplied course evidence, any accurate answer could be memory or chance. Inspect its claims before trusting them.

In [4]:
ask(QUESTION)

Okay, let's break down the assessment for UNSW COMP3110 (Algorithms and Data Structures).  The assessment has changed slightly over the years, so I'll give you the most current information as of the 2024 session (Session 3).  **Please always refer to the official COMP3110 course outline on Canvas for the definitive and most up-to-date details.**

Here's a breakdown of the assessment components, their weights, and what they generally involve:

**1. Assignments (40%)**

*   **Number:** Typically 4 assignments.
*   **Format:** Programming assignments. You'll be given problem descriptions and will need to write code (usually in C++) to solve them.
*   **Content:** Assignments cover the material taught in the lectures. They'll reinforce your understanding of algorithms and data structures. Expect topics like:
    *   Sorting and Searching
    *   Graph Algorithms (e.g., Dijkstra's, BFS, DFS)
    *   Dynamic Programming
    *   Trees (e.g., Binary Search Trees, Heaps)
    *   Hashing
*   **S

## 2: build the index

Split each PDF page into overlapping 200-word chunks, then embed every chunk. `nomic-embed-text` is trained with task prefixes (`search_document:` for chunks, `search_query:` for the question) and retrieval quietly degrades without them.

In [5]:
CHUNK_WORDS = 200
OVERLAP_WORDS = 40

def chunk_pdfs(paths):
    chunks = []
    for path in paths:
        for page_no, page in enumerate(PdfReader(path).pages, start=1):
            words = (page.extract_text() or "").split()
            step = CHUNK_WORDS - OVERLAP_WORDS
            for start in range(0, max(len(words), 1), step):
                text = " ".join(words[start : start + CHUNK_WORDS])
                if text.strip():
                    chunks.append({"source": f"{Path(path).name} p.{page_no}", "text": text})
    return chunks


def embed(texts, prefix="search_document"):
    result = client.embed(model=EMBED_MODEL, input=[f"{prefix}: {t}" for t in texts])
    return np.array(result["embeddings"])


chunks = chunk_pdfs([PDF])
chunk_vectors = embed([c["text"] for c in chunks])
print(f"{len(chunks)} chunks, vectors {chunk_vectors.shape}")

Ignoring wrong pointing object 92 0 (offset 0)


15 chunks, vectors (15, 768)


## Retrieve

Embed the question, score every chunk by cosine similarity, keep the top 4.

In [6]:
TOP_K = 4

def retrieve(question):
    q = embed([question], prefix="search_query")[0]
    scores = chunk_vectors @ q / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(q))
    ranked = np.argsort(scores)[::-1][:TOP_K]
    return [(chunks[i], scores[i]) for i in ranked]


hits = retrieve(QUESTION)
for chunk, score in hits:
    print(f"[{score:.3f}] {chunk['source']}: {chunk['text'][:110]}...")

[0.752] intro.pdf p.8: COMP3110 is an accelerator...
[0.663] intro.pdf p.5: COMP3110 vs COMP9417 Topics COMP3110 COMP9417 ML algorithms No Yes Theoretical foundations No Yes Datasets and...
[0.657] intro.pdf p.1: Course Intro Machine Learning Engineering COMP3110...
[0.606] intro.pdf p.13: Assessment Portfolio of Tasks (60%): evidence across onboarding and accelerator tasks, including individual co...


## Same question & model, with the retrieved chunks in the prompt

In [7]:
context = "\n\n---\n\n".join(f"[{c['source']}] {c['text']}" for c, _ in hits)
ask(QUESTION, context)

UNSW COMP3110 is assessed through a Portfolio of Tasks (60%) and a Final Project (40%) [intro.pdf p.13].